In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Concatenate, Embedding, Reshape, TimeDistributed, Lambda
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings("ignore")

# --- 1. Configuration ---
INPUT_FILE = '../../data/data_with_features.csv'
OUTPUT_FILE = '../../data/syntheticData/synthetic_data_diffusion.csv'
TARGET_COLUMN = 'cumulative_mite_count'
GROUP_COLUMN = 'treatment_id'

# Model & Diffusion Parameters
HIDDEN_UNITS = 64
EPOCHS = 500
BATCH_SIZE = 64
NUM_DIFFUSION_TIMESTEPS = 1000

# --- 2. Data Preparation (Unchanged) ---
print("--- ⚙️ Preparing Data for Diffusion Model ---")
df = pd.read_csv(INPUT_FILE)
grouped = df.groupby(GROUP_COLUMN)
original_series_list = [g[TARGET_COLUMN].values.astype(float) for _, g in grouped]
original_lengths = [len(s) for s in original_series_list]
max_len = max(original_lengths)

print(f"   Found {len(original_series_list)} time series groups.")
print(f"   Maximum series length is {max_len}.")

padded_series = np.array([np.pad(s, (0, max_len - len(s)), 'constant', constant_values=0) for s in original_series_list])
scaler = MinMaxScaler(feature_range=(-1, 1))
scaled_data = scaler.fit_transform(padded_series)
data_for_diffusion = scaled_data.reshape(len(scaled_data), max_len, 1)
print(f"   Data prepared with shape: {data_for_diffusion.shape}\n")

# --- 3. Diffusion Model Setup (Corrected) ---
print("--- ⚛️ Setting up Diffusion Components ---")

# Create the noise schedule (betas)
betas = tf.linspace(1e-4, 0.02, NUM_DIFFUSION_TIMESTEPS)
alphas = 1.0 - betas
alphas_cumprod = tf.math.cumprod(alphas)

# Helper function to get noisy data at any timestep `t`
def forward_noise(x0, t):
    noise = tf.random.normal(shape=tf.shape(x0))
    sqrt_alphas_cumprod_t = tf.gather(tf.sqrt(alphas_cumprod), t)
    sqrt_one_minus_alphas_cumprod_t = tf.gather(tf.sqrt(1.0 - alphas_cumprod), t)
    
    sqrt_alphas_cumprod_t = tf.reshape(sqrt_alphas_cumprod_t, [-1, 1, 1])
    sqrt_one_minus_alphas_cumprod_t = tf.reshape(sqrt_one_minus_alphas_cumprod_t, [-1, 1, 1])
    
    noisy_x = sqrt_alphas_cumprod_t * x0 + sqrt_one_minus_alphas_cumprod_t * noise
    return noisy_x, noise

# The denoising network (CORRECTED)
def build_denoising_network(timesteps, features):
    time_input = Input(shape=(), dtype=tf.int64, name='time_input')
    ts_input = Input(shape=(timesteps, features), name='ts_input')

    time_embedding = Embedding(NUM_DIFFUSION_TIMESTEPS, HIDDEN_UNITS)(time_input)
    time_embedding = Dense(HIDDEN_UNITS)(time_embedding)
    time_embedding = Reshape((1, HIDDEN_UNITS))(time_embedding)
    
    # --- THIS IS THE FIX ---
    # Wrap the tf.tile operation in a Lambda layer
    tiling_layer = Lambda(lambda x: tf.tile(x, [1, timesteps, 1]))
    time_embedding = tiling_layer(time_embedding)
    # ----------------------

    merged = Concatenate()([ts_input, time_embedding])
    gru1 = GRU(HIDDEN_UNITS, return_sequences=True)(merged)
    gru2 = GRU(HIDDEN_UNITS, return_sequences=True)(gru1)
    pred_noise = TimeDistributed(Dense(features))(gru2)

    return Model(inputs=[ts_input, time_input], outputs=pred_noise)

denoising_model = build_denoising_network(max_len, 1)
optimizer = Adam()
loss_fn = MeanSquaredError()
print("   Denoising network built successfully.\n")

# --- 4. Training Loop (Unchanged) ---
print(f"--- 🏋️ Training Diffusion Model for {EPOCHS} epochs ---")
dataset = tf.data.Dataset.from_tensor_slices(data_for_diffusion.astype(np.float32)).shuffle(len(data_for_diffusion)).batch(BATCH_SIZE)

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0
    for step, x_batch in enumerate(dataset):
        with tf.GradientTape() as tape:
            t = tf.random.uniform(minval=0, maxval=NUM_DIFFUSION_TIMESTEPS, shape=(len(x_batch),), dtype=tf.int32)
            noisy_batch, original_noise = forward_noise(x_batch, t)
            predicted_noise = denoising_model([noisy_batch, t], training=True)
            loss = loss_fn(original_noise, predicted_noise)
        
        gradients = tape.gradient(loss, denoising_model.trainable_weights)
        optimizer.apply_gradients(zip(gradients, denoising_model.trainable_weights))
        epoch_loss += loss.numpy()
        
    if epoch % 50 == 0:
        print(f"   Epoch {epoch}, Average Loss: {epoch_loss / (step + 1):.4f}")
print("   Training complete.\n")

# --- 5. Generation (Sampling) (Unchanged) ---
print("--- ✨ Generating New Synthetic Data (This will be slow) ---")
num_samples_to_generate = len(original_series_list)
generated_samples = tf.random.normal(shape=(num_samples_to_generate, max_len, 1))

for t in reversed(range(NUM_DIFFUSION_TIMESTEPS)):
    tt = tf.cast(tf.fill(num_samples_to_generate, t), dtype=tf.int64)
    predicted_noise = denoising_model.predict([generated_samples, tt], verbose=0, batch_size=BATCH_SIZE)
    
    alpha_t = alphas[t]
    alpha_t_cumprod = alphas_cumprod[t]
    
    coeff1 = 1 / tf.sqrt(alpha_t)
    coeff2 = (1 - alpha_t) / tf.sqrt(1 - alpha_t_cumprod)
    generated_samples = coeff1 * (generated_samples - coeff2 * predicted_noise)
    
    if t > 0:
        noise = tf.random.normal(shape=tf.shape(generated_samples))
        variance = tf.sqrt(betas[t])
        generated_samples += variance * noise
print("   Synthetic data generated.\n")

# --- 6. Post-processing and Saving (Unchanged) ---
print(f"--- 💾 Saving to {OUTPUT_FILE} ---")
generated_padded = scaler.inverse_transform(generated_samples.numpy().reshape(-1, max_len))

generated_final = []
for i, length in enumerate(original_lengths):
    series = generated_padded[i, :length]
    series = np.maximum(0, series)
    series = np.maximum.accumulate(series)
    generated_final.append(series)

synthetic_df_list = []
original_groups = [g for _, g in grouped]
for i, group_df in enumerate(original_groups):
    temp_df = group_df.copy()
    temp_df[TARGET_COLUMN] = generated_final[i]
    synthetic_df_list.append(temp_df)

final_synthetic_df = pd.concat(synthetic_df_list, ignore_index=True)
final_synthetic_df.to_csv(OUTPUT_FILE, index=False)
print(f"   Successfully saved synthetic data. Final shape: {final_synthetic_df.shape}")
print("\n--- ✅ Diffusion Model Process Complete ---")

--- ⚙️ Preparing Data for Diffusion Model ---
   Found 224 time series groups.
   Maximum series length is 29.
   Data prepared with shape: (224, 29, 1)

--- ⚛️ Setting up Diffusion Components ---
   Denoising network built successfully.

--- 🏋️ Training Diffusion Model for 500 epochs ---
   Epoch 50, Average Loss: 0.1183
   Epoch 100, Average Loss: 0.1062
   Epoch 150, Average Loss: 0.0792
   Epoch 200, Average Loss: 0.0757
   Epoch 250, Average Loss: 0.0752
   Epoch 300, Average Loss: 0.0658
   Epoch 350, Average Loss: 0.0707
   Epoch 400, Average Loss: 0.0621
   Epoch 450, Average Loss: 0.0540
   Epoch 500, Average Loss: 0.0574
   Training complete.

--- ✨ Generating New Synthetic Data (This will be slow) ---
   Synthetic data generated.

--- 💾 Saving to ../../data/syntheticData/synthetic_data_diffusion.csv ---
   Successfully saved synthetic data. Final shape: (3786, 14)

--- ✅ Diffusion Model Process Complete ---
